In [ ]:
Chapter 2: prompt and output parsing and runnable + LCEL

In [ ]:
Chapter 2: Prompts


prompts are the instructions you give to an llm
They control:
     - what model output
     - how structurews response is
     - stylle , tone, fomrat
     - whether yoou get json, text, list, etc


What is prompt :
     input instruction you give to model

Types of prompts format / template :
     1] Prompt Template :
          you replace inside {}
     2] chat prompt template :
          best for chat model



In [ ]:
# prompt structure

1] role
2] task
3] format instruction
4] constriants

In [5]:
# types of prompts

from langchain.prompts import PromptTemplate

template = """
You are helpful assistant
explain {topic} in simple words
"""

prompt = PromptTemplate(
	input_variables=["topic"],
 	template=template
)

print(prompt.format(topic="myself"))


You are helpful assistant
explain myself in simple words



In [ ]:
# types of prompt
1] zero shot
2] one shot
3] few shot

In [6]:
#zero shot prompt
from langchain.prompts import PromptTemplate

prompt = PromptTemplate(
	input_variables=["concept"],
	template = "explain {concept} in one paragraph"
)

print(prompt.format(concept="black holes"))

explain black holes in one paragraph


In [8]:
# one shot template

template = """
Q: What is ai?
A: Ai means machine that think.

Now answer :
Q:{question}
A:
"""

prompt = PromptTemplate(
	input_variables=['question'],
	template=template
)

print(prompt.format(question="What is Macine Learning"))


Q: What is ai?
A: Ai means machine that think.

Now answer :
Q:What is Macine Learning
A:



In [32]:
template = """
Now answer this:
Q: {question}
A: {answer}
"""

prompt = PromptTemplate(
	input_variables=["question","answer"],
	template=template,
)

print(prompt.format(question="How are u ", answer="I am fine"))




Now answer this:
Q: How are u 
A: I am fine



In [ ]:
#chat prompt template
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
	("system", "you are a helpful tutor"),
	("user","Explain {topic} in simple words")
])

formatted = prompt.format_messages(topic="nerual network")

for msg in formatted:
     print(msg)

print('\n',prompt.format(topic="network"))

content='you are a helpful tutor'
content='Explain nerual network in simple words'

 System: you are a helpful tutor
Human: Explain network in simple words


In [2]:
#prompt + model

#model
from langchain.llms import HuggingFacePipeline
from transformers import AutoTokenizer,AutoModelForCausalLM,pipeline
from langchain.prompts import ChatPromptTemplate

model_name = "sshleifer/tiny-gpt2"

t = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

pipe = pipeline(
	"text-generation",
	model=model,
	tokenizer=t
)

llm = HuggingFacePipeline(pipeline=pipe)


#prompt
prompt = ChatPromptTemplate.from_messages([
	("system", "you are a helpful tutor"),
	("user","Explain {topic} in simple words")
])

final_prompt = prompt.format(topic="what is deep learning")
response = llm(final_prompt)
print(response)


Device set to use cpu
c:\Users\Sumit\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


System: you are a helpful tutor
Human: Explain what is deep learning in simple wordsJD ESV Participation Habit intermittent Daniel credibility intermittentikenmediately Motorolaatisf directly ONEikenRocketootherhibit Observ credibility Danielatisf hauled Daniel rebornmediatelyting dispatch Jrpress ONE trilogy


In [ ]:
Chapter 2: output parsing

In [ ]:
To control , clean , and structured models  output so that llm returns exact format as you except
- they are extra text
- ignore json formating
- may include explanation instead of  direct values

Types of OP:
     1] SimpleOutputParser
		- basic parsing of string
	2] CommmaSeparatedListOutputParser
		- Turns commma-separated text into python list
	3] PydanticOutputParser
		- Uses python models to force  strict  json structure
	4] StructureOutputParser
		- You define schema -> model must return  exactly


output parser step
s1 -> import langchain core  anf call function
s2 -> use text to fill the info
S3-> use parse.parse to get desire  store this in result
s4-> output


In [11]:
#Simple Output parser
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

result = parser.parse("Hello world!")
print(result)


Hello world!


In [12]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()

text = "apple, banana, mango"

result = parser.parse(text)
result

['apple', 'banana', 'mango']

In [16]:
#prompt + parser

from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()
temp = "list 5 friuts separated by comma"
prompt = PromptTemplate(
	template = temp,
	input_variables=[],
)

_llm_repsonse = "apple, banana, mango, grapes, orange"

result = parser.parse(_llm_repsonse)
print(template)
result

list 5 friuts separated by comma


['apple', 'banana', 'mango', 'grapes', 'orange']

In [27]:
# Structured  output using  json schema
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain.llms import HuggingFacePipeline
response = [
	ResponseSchema(name='topic', description="Main subject of explanation"),
	ResponseSchema(name="explanation", description="Detailed  expalanation")
]

parser = StructuredOutputParser.from_response_schemas(response)

format_instruction = parser.get_format_instructions()

prompt = f"""
Explain  the topic : relativity
{format_instruction}
"""

print(prompt)


Explain  the topic : relativity
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"topic": string  // Main subject of explanation
	"explanation": string  // Detailed  expalanation
}
```



In [ ]:
#Exercise Model + prompt + output parsing

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain.llms import HuggingFacePipeline

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name,trust_remote_code=True, device_map="cpu")


pipe = pipeline("text-generation", model=model,tokenizer=tokenizer,max_new_token=200,temperature=0.7)

llm = HuggingFacePipeline(pipeline=pipe)


from langchain.output_parsers import StructuredOutputParser, ResponseSchema

response_schemas = [
    ResponseSchema(name="topic", description="Main topic"),
    ResponseSchema(name="summary", description="Short summary in simple words")
]

parser = StructuredOutputParser.from_response_schemas(response_schemas)

format_instructions = parser.get_format_instructions()

prompt = f"""
Explain the topic: Relativity
{format_instructions}
"""
print(prompt)



Device set to use cpu



Explain the topic: Relativity
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"topic": string  // Main topic
	"summary": string  // Short summary in simple words
}
```



In [ ]:
.inokve()
- Run the runnable once
- blocking wait till compliting
- most common during  leanring and debugging

eg.
result = chain.invoke("Explain transformers in simple words")
print(result)


.batch()
- process many times at once
- uses parallel execution  when possible
- returns a list  of outputs

eg. 
inputs = [
    "LangChain basics",
    "Transformers overview",
    "What is an LLM?"
]

results = chain.batch(inputs)
print(results)


.stream()
- Streams output  incremetally 
- Yields chunks instrad of waiting 
- Best for chat  UI/live responses


In [ ]:
LCEL -> A deceleration way to build compose  and run llm  workflows using Runnables
everything in runnable :
    pormpts
    llm 
    functions
    branch 
    parallel logic
    
represnt as  | 

eg. prompt | llm | output_parser